# 5. Linear Regression
**Use case:** Insurance — Auto insurance expected claim cost and premium pricing

An auto insurer must quote a price for every driver before knowing whether they will crash. Charge too little and the book loses money; charge too much and good drivers leave for competitors. Rating data includes driver age, miles driven, vehicle value, prior claims, territory and (where allowed) credit-based insurance score, plus a year of claims history. Success means a pure-premium model whose predictions balance to actual losses, rank risk well, stay positive and can be explained factor-by-factor in a rate filing.

> 💡 A taxi fare is a base charge plus a rate per mile plus a rate per minute. Linear regression looks at many past trips and works out the base charge and the rates that best explain the fares you actually paid.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Fit a pricing line through 10 drivers
Solve least squares with numpy to predict yearly claim cost from driver age and miles driven, then price a new policy.

| 5W + H | |
|---|---|
| **Who** | A pricing analyst or underwriter learning how a rating formula is estimated. |
| **What** | A two-variable linear formula for expected annual claim cost (the pure premium) and a quoted premium. |
| **When** | At quote time - every new auto policy needs an expected loss cost before expenses and profit are added. |
| **Where** | On a laptop with numpy; the same idea sits inside every rating engine's formula. |
| **Why** | Linear regression is the simplest way to turn risk factors into dollars and read each factor's effect directly. |
| **How** | Store 10 policies as dicts, build a design matrix with an intercept column, solve with np.linalg.lstsq, print errors and R-squared. |

**🧠 ML concepts:** Least squares, Intercept and slopes, Residuals / errors, R-squared, Pure premium vs charged premium  
**🐍 Python concepts:** List of dicts, List comprehensions to extract a field, np.column_stack, Star-unpacking with coef, *_, Matrix multiplication with @, zip over four arrays

In [ ]:
# EASY: linear regression by hand - expected yearly claim cost for an auto policy
import numpy as np

# 🐍 PY: a list of dicts - each dict is one policyholder with named fields
policies = [
    {"driver_age": 19, "annual_miles_k": 14, "claim_cost": 1650},
    {"driver_age": 22, "annual_miles_k": 12, "claim_cost": 1380},
    {"driver_age": 25, "annual_miles_k": 15, "claim_cost": 1290},
    {"driver_age": 30, "annual_miles_k": 10, "claim_cost": 900},
    {"driver_age": 35, "annual_miles_k": 12, "claim_cost": 870},
    {"driver_age": 40, "annual_miles_k": 8, "claim_cost": 640},
    {"driver_age": 45, "annual_miles_k": 11, "claim_cost": 700},
    {"driver_age": 50, "annual_miles_k": 9, "claim_cost": 560},
    {"driver_age": 55, "annual_miles_k": 13, "claim_cost": 690},
    {"driver_age": 60, "annual_miles_k": 7, "claim_cost": 480},
]
# 🐍 PY: list comprehensions pull one field out of every dict
age = np.array([p["driver_age"] for p in policies], dtype=float)
miles = np.array([p["annual_miles_k"] for p in policies], dtype=float)
cost = np.array([p["claim_cost"] for p in policies], dtype=float)

# 🧠 ML: linear regression = find the straight-line formula cost = b0 + b1*age + b2*miles with least error
# 🐍 PY: np.column_stack builds the design matrix; the column of ones lets the model learn an intercept b0
X = np.column_stack([np.ones(len(age)), age, miles])
# 🧠 ML: least squares picks the coefficients that minimise the sum of squared errors
coef, *_ = np.linalg.lstsq(X, cost, rcond=None)
b0, b_age, b_miles = coef
print(f"cost = {b0:.0f} {b_age:+.1f} x age {b_miles:+.1f} x miles_k")
print(f"-> each extra year of age changes expected cost by ${b_age:.1f}")
print(f"-> each extra 1,000 miles driven adds ${b_miles:.1f}\n")

# 🐍 PY: @ multiplies the matrix by the coefficient vector -> one prediction per row
pred = X @ coef
print(" age | miles_k | actual | predicted | error")
# 🐍 PY: zip walks four arrays together in one loop
for a, m, c, p in zip(age, miles, cost, pred):
    print(f"{a:>4.0f} | {m:>7.0f} | {c:>6.0f} | {p:>9.0f} | {c - p:>+5.0f}")

# 🧠 ML: R-squared = share of the variation in cost explained by the model (1.0 = perfect)
# 🧠 ML: notice errors are positive at BOTH ends (age 19 and 60) - a curve, not a line, fits risk vs age
r2 = 1 - np.sum((cost - pred) ** 2) / np.sum((cost - cost.mean()) ** 2)
print(f"\nR-squared: {r2:.3f}")

# Price a new policy: pure premium = expected claim cost, then add a 25% expense/profit load
new_driver = np.array([1, 28, 16])
pure = new_driver @ coef
print(f"New 28-year-old driving 16k miles: expected claims ${pure:.0f}, premium ${pure * 1.25:.0f}")

**Try this next:**
- Add an age-squared column (age ** 2) to X and see whether the errors at both ends shrink.
- Remove miles from the model and compare R-squared.
- Price a 70-year-old driver - is a negative age slope sensible that far out?

## MEDIUM: Pure premium model on 4,000 policies
Fit a scikit-learn linear regression with one-hot rating factors, read dollar coefficients and spot where a straight line misprices.

| 5W + H | |
|---|---|
| **Who** | A pricing actuary or analyst building a first-cut loss-cost model for a personal auto book. |
| **What** | A linear model for expected annual claim cost with dollar effects for each rating factor and quotes for new drivers. |
| **When** | During the annual rate review, using last year's policy and claims data. |
| **Where** | In the pricing team's analytics environment; results inform the rating plan filed with the state. |
| **Why** | Coefficients in dollars are easy to explain to underwriters and regulators, and residual checks show where the model is too simple. |
| **How** | Simulate 4,000 policies, one-hot encode territory and credit tier in a ColumnTransformer, fit LinearRegression, report MAE and R-squared, print coefficients and residuals by age band. |

**🧠 ML concepts:** Multiple linear regression, One-hot encoding with a reference level, Interpreting coefficients holding others fixed, MAE and R-squared, Residual analysis to find non-linearity  
**🐍 Python concepts:** Series.map with a dict, np.maximum element-wise, ColumnTransformer and Pipeline, str.split to clean names, itertuples for fast row loops, zip over iterrows and predictions

In [ ]:
# MEDIUM: scikit-learn linear regression for expected auto claim cost (pure premium) per policy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(42)
n = 4000
df = pd.DataFrame({
    "driver_age": rng.integers(18, 80, n),
    "annual_miles_k": rng.normal(12, 4, n).clip(2, 30).round(1),
    "vehicle_value_k": rng.lognormal(3.2, 0.4, n).round(1),
    "prior_claims_3y": rng.poisson(0.3, n),
    "credit_tier": rng.choice(["A", "B", "C"], n, p=[.5, .35, .15]),
    "territory": rng.choice(["urban", "suburban", "rural"], n, p=[.35, .45, .2]),
})
# Hidden truth: young drivers costlier, more miles, pricier car, prior claims, urban territory
# 🐍 PY: np.maximum compares element-wise - drivers over 30 get 0 extra risk
young = np.maximum(0, 30 - df.driver_age)
df["claim_cost"] = (420 + 28 * young + 22 * df.annual_miles_k + 6 * df.vehicle_value_k
                    + 210 * df.prior_claims_3y + df.territory.map({"urban": 260, "suburban": 90, "rural": 0})
                    + df.credit_tier.map({"A": 0, "B": 70, "C": 160}) + rng.normal(0, 180, n)).round(0)
print(f"Policies: {n}  mean annual claim cost: ${df.claim_cost.mean():,.0f}")

num = ["driver_age", "annual_miles_k", "vehicle_value_k", "prior_claims_3y"]
cat = ["credit_tier", "territory"]
X_tr, X_te, y_tr, y_te = train_test_split(df[num + cat], df.claim_cost, test_size=0.25, random_state=42)

# 🧠 ML: one-hot turns each category into 0/1 columns; drop='first' avoids perfectly redundant columns
# 🐍 PY: a ColumnTransformer sends different columns to different preprocessing steps
pre = ColumnTransformer([("num", "passthrough", num),
                         ("cat", OneHotEncoder(drop="first"), cat)])
model = Pipeline([("pre", pre), ("lr", LinearRegression())])
model.fit(X_tr, y_tr)

# 🧠 ML: MAE = average dollar miss per policy; R-squared = share of cost variation explained
pred = model.predict(X_te)
print(f"Test MAE ${mean_absolute_error(y_te, pred):,.0f} | R-squared {r2_score(y_te, pred):.3f}")

# 🧠 ML: each coefficient is the $ change in expected cost for +1 unit, holding everything else fixed
names = model.named_steps["pre"].get_feature_names_out()
coefs = model.named_steps["lr"].coef_
print(f"\nIntercept: ${model.named_steps['lr'].intercept_:,.0f}")
# 🐍 PY: str.split('__')[1] strips the 'num__' / 'cat__' prefix added by ColumnTransformer
for nm, c in zip(names, coefs):
    print(f"  {nm.split('__')[1]:<18} {c:>+8.1f} $")

# 🧠 ML: driver_age is linear here, but true risk only rises below 30 - residuals by age band reveal it
res = X_te.assign(actual=y_te.values, pred=pred)
res["age_band"] = pd.cut(res.driver_age, [17, 25, 35, 50, 80])
check = res.groupby("age_band", observed=True)[["actual", "pred"]].mean()
print("\nAge band | actual $ | predicted $ | gap")
# 🐍 PY: itertuples is a fast way to loop rows as named tuples
for row in check.itertuples():
    print(f"{str(row.Index):<9}| {row.actual:>8.0f} | {row.pred:>11.0f} | {row.actual - row.pred:>+5.0f}")

# Business: pure premium + 30% expense & profit load for two quotes
# 🐍 PY: a small DataFrame of new quotes must use the same column names as the training data
quotes = pd.DataFrame({"driver_age": [20, 45], "annual_miles_k": [15, 10], "vehicle_value_k": [18, 35],
                       "prior_claims_3y": [1, 0], "credit_tier": ["B", "A"], "territory": ["urban", "rural"]})
for (_, q), pp in zip(quotes.iterrows(), model.predict(quotes)):
    print(f"Quote age {q.driver_age}, {q.territory}: pure premium ${pp:,.0f} -> charged ${pp * 1.3:,.0f}")

**Try this next:**
- Add a 'young' column = max(0, 30 - age) to the features and watch the age-band gaps close.
- Remove credit_tier and see how the other coefficients change (omitted-variable effect).
- Try sklearn's Ridge(alpha=10) and compare coefficients and MAE.

## HARD: Rate plan: OLS vs splines vs Tweedie GLM
Compare plain OLS, Ridge with age splines and a tuned Tweedie GLM on an exposure-weighted policy book, then produce relativities, an age curve and lift.

| 5W + H | |
|---|---|
| **Who** | Pricing actuaries, the chief actuary sign-off, and state regulators who review rate filings. |
| **What** | A saved Tweedie GLM pricing model with multiplicative rating relativities, an indicated age curve, balance and decile lift checks. |
| **When** | Annual or semi-annual rate review, using a full year of policy exposure and claims. |
| **Where** | In the actuarial modelling environment; relativities flow into the rating engine and the rate filing. |
| **Why** | Most policies have $0 claims and a few are very large; OLS can predict negative premiums, while the Tweedie GLM with log link is the actuarial standard for pure premium. |
| **How** | Simulate 20,000 policies with Poisson frequency and gamma severity, build three pipelines, tune Tweedie power and alpha with exposure weights, compare MAE, deviance and Gini, then extract relativities and save. |

**🧠 ML concepts:** Ordinary least squares vs generalised linear models, Tweedie distribution (compound Poisson-gamma), Log link and multiplicative relativities, Exposure weights, Spline features for non-linear effects, Gini / Lorenz lift and balance checks  
**🐍 Python concepts:** List comprehension with a conditional expression, Passing fit params as step__param, Nested ColumnTransformer with remainder, String slicing to clean names, DataFrame.iloc with a repeated index, pd.qcut(labels=False)

In [ ]:
# HARD: auto pure-premium pricing - OLS vs engineered Ridge vs Tweedie GLM (actuarial standard)
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, TweedieRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_tweedie_deviance

warnings.filterwarnings("ignore")
# 🐍 PY: constants at the top make it obvious which columns the pipeline expects
NUM = ["driver_age", "annual_miles_k", "vehicle_value_k", "prior_claims_3y", "years_licensed"]
CAT = ["territory", "vehicle_use", "credit_tier"]


def make_book(n=20000, seed=42):
    """Simulate a policy book: most policies have $0 claims, a few have large losses."""
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 85, n)
    df = pd.DataFrame({
        "driver_age": age,
        "years_licensed": np.clip(age - 17 - rng.integers(0, 5, n), 0, None),
        "annual_miles_k": rng.normal(12, 4, n).clip(2, 30).round(1),
        "vehicle_value_k": rng.lognormal(3.2, 0.4, n).round(1),
        "prior_claims_3y": rng.poisson(0.25, n),
        "territory": rng.choice(["urban", "suburban", "rural"], n, p=[.35, .45, .2]),
        "vehicle_use": rng.choice(["commute", "pleasure", "business"], n, p=[.6, .3, .1]),
        "credit_tier": rng.choice(["A", "B", "C"], n, p=[.5, .35, .15]),
        "exposure": rng.uniform(0.25, 1.0, n).round(2),   # fraction of a policy-year in force
    })
    # 🧠 ML: frequency is multiplicative (log-linear) with a U-shaped age curve - typical for auto
    log_freq = (np.log(0.08) + 1.2 * np.exp(-(age - 18) / 8) + 0.6 * np.clip(age - 70, 0, None) / 15
                + 0.05 * (df.annual_miles_k - 12) + 0.4 * df.prior_claims_3y
                + df.territory.map({"urban": 0.5, "suburban": 0.1, "rural": 0.0})
                + df.vehicle_use.map({"commute": 0.1, "pleasure": 0.0, "business": 0.3})
                + df.credit_tier.map({"A": 0.0, "B": 0.2, "C": 0.45}))
    # 🧠 ML: claim counts are Poisson, scaled by exposure - half a year in force = half the expected claims
    counts = rng.poisson(np.exp(log_freq) * df.exposure)
    # Severity: gamma-distributed $ per claim, larger for expensive vehicles
    sev_mean = 4200 * (df.vehicle_value_k / 25) ** 0.3
    # 🐍 PY: a list comprehension with a conditional sums k random severities only where k > 0
    loss = [rng.gamma(2.0, m / 2.0, k).sum() if k else 0.0 for k, m in zip(counts, sev_mean)]
    df["pure_premium"] = np.array(loss) / df.exposure   # $ per full policy-year
    return df


def preprocessor(splines=False):
    num_step = StandardScaler()
    if splines:
        # 🧠 ML: splines let a LINEAR model bend - key to capturing the U-shaped age curve
        num_step = ColumnTransformer([("age", SplineTransformer(n_knots=5, degree=3), ["driver_age"])],
                                     remainder=StandardScaler())
    return ColumnTransformer([("num", num_step, NUM), ("cat", OneHotEncoder(drop="first"), CAT)])


def lorenz_gini(y, pred, w):
    """Ordered Lorenz / Gini - how well the model ranks risk (actuaries' favourite lift metric)."""
    # 🧠 ML: sort policies from safest to riskiest predicted, then see how fast actual losses accumulate
    order = np.argsort(pred)
    cum_loss = np.cumsum((y * w)[order]) / np.sum(y * w)
    cum_expo = np.cumsum(w[order]) / np.sum(w)
    return 1 - 2 * np.trapezoid(cum_loss, cum_expo)


def main():
    df = make_book()
    print(f"Policies {len(df):,} | with a claim {np.mean(df.pure_premium > 0):.1%} | "
          f"avg pure premium ${np.average(df.pure_premium, weights=df.exposure):,.0f}")
    X, y, w = df[NUM + CAT], df.pure_premium.values, df.exposure.values
    X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.25, random_state=42)

    # 🧠 ML: OLS predicts the mean directly; it can go negative and treats a $0 year like any other value
    models = {
        "OLS (plain linear)": Pipeline([("pre", preprocessor()), ("m", LinearRegression())]),
        "Ridge + age splines": Pipeline([("pre", preprocessor(True)), ("m", RidgeCV(alphas=[0.1, 1, 10, 100]))]),
    }
    # 🧠 ML: Tweedie GLM with log link: power 1-2 = compound Poisson-gamma, handles the pile of $0 claims
    glm = Pipeline([("pre", preprocessor(True)), ("m", TweedieRegressor(link="log", max_iter=1000))])
    grid = {"m__power": [1.3, 1.5, 1.7], "m__alpha": [0.0001, 0.001, 0.01]}
    # 🐍 PY: fit params for a pipeline step are passed as <step>__<param>, here the exposure weights
    gs = GridSearchCV(glm, grid, cv=KFold(3, shuffle=True, random_state=42),
                      scoring="neg_mean_poisson_deviance", n_jobs=1).fit(X_tr, y_tr, m__sample_weight=w_tr)
    print("Tweedie best:", {k[3:]: v for k, v in gs.best_params_.items()})
    models["Tweedie GLM (log link)"] = gs.best_estimator_

    print("\nModel                   MAE $   Tweedie dev   Gini  min pred $")
    preds = {}
    for name, m in models.items():
        if name != "Tweedie GLM (log link)":
            m.fit(X_tr, y_tr, m__sample_weight=w_tr)
        p = m.predict(X_te)
        preds[name] = p
        # 🧠 ML: Tweedie deviance needs positive predictions - OLS can predict negative premiums!
        dev = mean_tweedie_deviance(y_te, np.clip(p, 1, None), power=1.5, sample_weight=w_te)
        print(f"{name:<23} {mean_absolute_error(y_te, p, sample_weight=w_te):>6,.0f} {dev:>12.1f} "
              f"{lorenz_gini(y_te, p, w_te):>6.3f} {p.min():>10,.0f}")

    best = models["Tweedie GLM (log link)"]
    p = preds["Tweedie GLM (log link)"]
    # 🧠 ML: balance check - total predicted premium should match total actual losses on test
    print(f"\nBalance: predicted ${np.sum(p * w_te):,.0f} vs actual ${np.sum(y_te * w_te):,.0f}")

    # 🧠 ML: log link -> exp(coef) is a multiplicative rating relativity, exactly how rate filings read
    names = best.named_steps["pre"].get_feature_names_out()
    coefs = best.named_steps["m"].coef_
    print("\nRating relativities (categorical factors):")
    for nm, c in zip(names, coefs):
        if nm.startswith("cat__"):
            print(f"  {nm[5:]:<20} x{np.exp(c):.2f}")

    # Age curve the GLM learned, via predictions for a reference driver
    ref = X_te.iloc[[0] * 6].copy()
    ref["driver_age"] = [18, 22, 30, 45, 65, 80]
    ref["years_licensed"] = (ref.driver_age - 17).clip(0, None)
    print("\nAge | indicated pure premium (reference driver)")
    for a, v in zip(ref.driver_age, best.predict(ref)):
        print(f"{a:>3} | ${v:>7,.0f} {'#' * int(v / 100)}")

    # Decile lift: does the riskiest tenth really cost more?
    # 🐍 PY: qcut with labels=False returns decile numbers 0-9 instead of interval labels
    dec = pd.qcut(p, 10, labels=False)
    lift = pd.DataFrame({"d": dec, "loss": y_te * w_te, "w": w_te}).groupby("d").sum()
    lr = (lift.loss / lift.w).values
    print(f"\nActual loss cost, lowest vs highest predicted decile: ${lr[0]:,.0f} vs ${lr[-1]:,.0f} ({lr[-1] / lr[0]:.1f}x)")

    print("\nRegulatory: rates must not be excessive, inadequate or unfairly discriminatory (state rating laws);")
    print("credit-based factors are restricted in some states; NAIC AI Bulletin expects governance + testing.")
    path = os.path.join(tempfile.mkdtemp(), "auto_pure_premium_glm.joblib")
    joblib.dump(best, path)
    print(f"Saved {os.path.basename(path)}")


if __name__ == "__main__":
    main()

**Try this next:**
- Fix power=1.0 (Poisson) and 1.9 (near gamma) and compare deviance and Gini.
- Remove the spline step (preprocessor(False)) for the GLM and look at the age curve.
- Drop credit_tier (banned in some states) and see how much lift and balance change.